In [1]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed_v3"
demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2003_2004_raw_corrected.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2003_2004_raw_corrected.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_questionnaire_2003_2004_raw_corrected.csv"
Examination_file = DATA_DIR / "neighbourhood_nhanes_examination_2003_2004_raw_corrected.csv"
Dietary_file = DATA_DIR / "neighbourhood_nhanes_dietary_2003_2004_raw_corrected.csv"


# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file, dtype=str, keep_default_na=False)
labs = pd.read_csv(labs_file, dtype=str, keep_default_na=False)
questionnaire = pd.read_csv(questionnaire_file, dtype=str, keep_default_na=False)
Examination = pd.read_csv(Examination_file, dtype=str, keep_default_na=False)
Dietary = pd.read_csv(Dietary_file, dtype=str, keep_default_na=False)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)
print("Examination:", Examination.shape)
print("Dietary:", Dietary.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================
demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()
Examination.columns = Examination.columns.str.upper()
Dietary.columns = Dietary.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================
summary = []
for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])

final_df = (
    demo
    .merge(questionnaire, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_QUESTIONNAIRE"))
    .merge(labs, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_LAB"))
    .merge(Examination, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_EXAMINATION"))
    .merge(Dietary, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_DIETARY"))
)
# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", final_df.shape)
print("Unique SEQN Count:", final_df["SEQN"].nunique())
print("Duplicate SEQN Count:", final_df.duplicated(subset=["SEQN"]).sum())
display(final_df.head(20))

OUTPUT_DIR = BASE_DIR / "data" / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2003_2004_corrected.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)


📊 Loaded Files:
Demographics: (10122, 46)
Labs: (9179, 30)
Questionnaire: (10122, 441)
Examination: (9643, 33)
Dietary: (9643, 240)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire
✅ SEQN found in Examination
✅ SEQN found in Dietary


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,10122,46,10122,0
1,labs,9179,30,9179,0
2,questionnaire,10122,441,10122,0
3,Examination,9643,33,9643,0
4,Dietary,9643,240,9643,0


📊 Final Unified Dataset Shape: (10122, 786)
Unique SEQN Count: 10122
Duplicate SEQN Count: 0


,SEQN,SDDSRVYR,RIDSTATR,RIDEXMON,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDAGEEX,RIDRETH1,RIDRETH2,...,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320,DR2_330,DR2BWATR,DR2CWATR,DR2TWS
0,21005,3,2,1,1,19,232,233,4,2,...,0.179,0.0,0.014,0.019,2,518.0,518.0,0.0,0.0,1
1,21006,3,2,2,2,16,203,205,4,2,...,0.016,0.0,0.0,0.0,1,947.2,947.2,0.0,0.0,1
2,21007,3,2,1,2,14,172,172,3,1,...,0.028999999999999998,0.01,0.0,0.006,2,592.0,0.0,592.0,0.0,
3,21008,3,2,2,1,17,208,209,4,2,...,0.045,0.0,0.0,0.002,2,858.4,858.4,0.0,0.0,1
4,21009,3,2,2,1,55,671,672,3,1,...,0.192,0.004,0.0,0.019,2,621.6,621.6,0.0,0.0,1
5,21010,3,2,2,2,52,633,634,3,1,...,0.026,0.0,0.0,0.001,3,858.4,858.4,0.0,0.0,2
6,21011,3,2,1,1,0,3,4,1,3,...,,,,,2,7.4,7.4,0.0,0.0,1
7,21012,3,2,2,1,63,765,766,4,2,...,0.243,0.003,0.0,0.012,2,0.0,0.0,0.0,0.0,
8,21013,3,2,1,2,13,163,164,4,2,...,0.033,0.0,0.0,0.001,2,355.2,0.0,355.2,0.0,
9,21014,3,2,2,1,3,42,42,4,2,...,0.047,0.0,0.0,0.003,2,592.0,592.0,0.0,0.0,1



✅ Final unified NHANES dataset saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\neighbourhood_nhanes_unified_2003_2004_corrected.csv
